Predict CMEs based on solar active regions and flare intensity assignment (A2)
==============================================================================
Collecting, cleaning, and pruning data
--------------------------------------

In [ ]:
pip install sunpy[all]

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 39.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 61.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 65.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 738.7/738.7 kB 49.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 966.4/966.4 kB 53.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 60.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 63.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 80.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 69.5 MB/s  0:00:00
  Created wheel for asciitree: filename=asciitree-0.3.3-py3-none-any.whl size=5095 sha256=76b6e8730861282119aeda8f44f9ca8444565a774f590544ea66a09d526561e8
  Stored in directory: /home/yangz

In [2]:
pip install requests

Note: you may need to restart the kernel to use updated packages.


In [3]:
import numpy as np
import matplotlib.pylab as plt
import matplotlib.mlab as mlab
import pandas as pd
import scipy.stats
import requests
import urllib
import json
from datetime import datetime as dt_obj
from datetime import timedelta
from sklearn import svm
from sklearn.model_selection import StratifiedKFold
from sunpy.time import TimeRange
from sunpy.net import Fido, attrs as a
import json

In [4]:
t_start = "2010-05-01"
t_end = "2015-05-01"

First we fetch the data for CMEs from DONKI.

In [5]:
# Location of the DONKI server
url = "https://kauai.ccmc.gsfc.nasa.gov/DONKI/WS/get/FLR?"+"startDate="+t_start+"&endDate="+t_end
    
# Is it up and running?
response = requests.get(url)
if response.status_code != 200:
    print('cannot successfully get an http response')

In [6]:
# Fetch data
print("Getting data from", url)
df = pd.read_json(url)

# select flares associated with a linked event (SEP or CME), and
# select only M or X-class flares
events_list = df.loc[df['classType'].str.contains(
    "M|X") & ~df['linkedEvents'].isnull()]

# drop all rows that don't satisfy the above conditions
events_list = events_list.reset_index(drop=True)

# Stats
print("Raw DONKI returned ",events_list.shape[0],"SEP or CME associated flares.")

Getting data from https://kauai.ccmc.gsfc.nasa.gov/DONKI/WS/get/FLR?startDate=2010-05-01&endDate=2015-05-01
Raw DONKI returned  121 SEP or CME associated flares.


In [7]:
# drop the rows that aren't linked to CME events
for i in range(events_list.shape[0]):
    value = events_list.loc[i]['linkedEvents'][0]['activityID']
    if not "CME" in value:
        #print(value, "not a CME, dropping row")
        events_list = events_list.drop([i])
events_list = events_list.reset_index(drop=True)
print("Raw DONKI returned ",events_list.shape[0],"CME associated flares.")

Raw DONKI returned  116 CME associated flares.


Convert the peakTime column in the events_list dataframe from a string into a datetime object:

In [8]:
def parse_tai_string(tstr):
    year = int(tstr[:4])
    month = int(tstr[5:7])
    day = int(tstr[8:10])
    hour = int(tstr[11:13])
    minute = int(tstr[14:16])
    return dt_obj(year, month, day, hour, minute)


for i in range(events_list.shape[0]):
    events_list.loc[i,'peakTime'] = parse_tai_string(
        events_list.loc[i,'peakTime'])

Next we fetch the GOES data describing the X-ray class of the observed flares

In [9]:
results = Fido.search(a.Time(t_start,t_end),
                            a.hek.EventType("FL"),
                            a.hek.FL.GOESCls > "M1.0",a.hek.OBS.Observatory == "GOES")

In [10]:
hek_results = results["hek"]
print("From GOES we found",len(hek_results[:]['ar_noaanum']),"events.")

From GOES we found 587 events.


In [11]:
# Case 1: CME and Flare exist but NOAA active region number does not exist in DONKI database

number_of_donki_mistakes = 0  # count the number of DONKI mistakes
# create an empty array to hold row numbers to drop at the end
event_list_drops = []

for i in range(events_list.shape[0]):
    if (np.isnan(events_list.loc[i]['activeRegionNum'])):
        time = events_list['peakTime'].iloc[i]
        shortres = Fido.search(a.Time(time,time),
                                a.hek.EventType("FL"),
                                a.hek.FL.GOESCls > "M1.0",
                                a.hek.OBS.Observatory == "GOES")
        listofresults=shortres["hek"]
        if (len(listofresults) == 0):
            print(events_list.loc[i]['activeRegionNum'], events_list.loc[i]
                    ['classType'], "has no match in the GOES flare database ; dropping row.")
            event_list_drops.append(i)
            number_of_donki_mistakes += 1
            continue
        else:
            print("Missing NOAA number:", events_list.loc[i,'activeRegionNum'], events_list.loc[i,'classType'],
                     events_list.loc[i,'peakTime'], "should be", listofresults[0]['ar_noaanum'], "; changing now.")
            events_list.loc[i,'activeRegionNum'] = listofresults[0]['ar_noaanum']
            number_of_donki_mistakes += 1

# Drop the rows for which there is no active region number in both the DONKI and GOES flare databases
events_list = events_list.drop(event_list_drops)
events_list = events_list.reset_index(drop=True)
print('There are', number_of_donki_mistakes, 'DONKI mistakes so far.')

Missing NOAA number: nan X1.4 2011-09-22 11:01:00 should be 11302 ; changing now.
Missing NOAA number: nan X1.3 2012-03-07 01:14:00 should be 11430 ; changing now.
Missing NOAA number: nan M6.3 2012-03-09 03:53:00 should be 11429 ; changing now.
Missing NOAA number: nan M5.1 2012-05-17 01:47:00 should be 11476 ; changing now.
Missing NOAA number: nan X1.1 2012-07-06 23:08:00 should be 11515 ; changing now.
Missing NOAA number: nan M6.2 2012-07-28 20:56:00 should be 11532 ; changing now.
Missing NOAA number: nan M1.7 2012-11-08 02:23:00 should be 11611 ; changing now.
Missing NOAA number: nan M1.2 2013-03-15 06:58:00 should be 11692 ; changing now.
Missing NOAA number: nan X1.6 2013-05-13 02:17:00 should be 11748 ; changing now.
Missing NOAA number: nan X2.8 2013-05-13 16:05:00 should be 11748 ; changing now.
Missing NOAA number: nan X3.2 2013-05-14 01:11:00 should be 11748 ; changing now.
Missing NOAA number: nan X1.2 2013-05-15 01:48:00 should be 11748 ; changing now.
Missing NOAA num

Case 2: NOAA active region is wrong in the DONKI data:

In [12]:
# collect all the peak flares times in the NOAA database
#peak_times_noaa = [item["event_peaktime"] for item in hek_results]
peak_times_noaa = hek_results[:]["event_peaktime"]

for i in range(events_list.shape[0]):
    # check if a particular DONKI flare peak time is also in the NOAA database
    # peak_time_donki = events_list['peakTime'].iloc[i]
    peak_time_donki = events_list.loc[i,'peakTime']
    idx = np.where(peak_time_donki == peak_times_noaa)[0]
    if len(idx)==0:
        continue
        
    # ignore NOAA active region numbers equal to zero
    if (hek_results[idx[0]]['ar_noaanum'] == 0):
        continue
    # if yes, check if the DONKI and NOAA active region numbers match up for this peak time
    # if they don't, flag this peak time and replace the DONKI number with the NOAA number
    if (hek_results[idx[0]]['ar_noaanum'] != int(events_list.loc[i,'activeRegionNum'])):
        print('Messed up NOAA number:', int(events_list.loc[i,'activeRegionNum']), events_list.loc[i,'classType'],
                events_list.loc[i,'peakTime'], "should be", hek_results[idx[0]]['ar_noaanum'], "; changing now.")
        events_list.loc[i,'activeRegionNum'] = hek_results[idx[0]]['ar_noaanum']
        number_of_donki_mistakes += 1
print('There are', number_of_donki_mistakes, 'DONKI mistakes so far.')

Messed up NOAA number: 11943 X1.2 2014-01-07 18:32:00 should be 11944 ; changing now.
Messed up NOAA number: 12051 M1.2 2014-05-07 16:29:00 should be 12055 ; changing now.
Messed up NOAA number: 12160 M1.4 2014-07-01 11:23:00 should be 12106 ; changing now.
Messed up NOAA number: 12282 M2.4 2015-02-09 23:35:00 should be 12280 ; changing now.
Messed up NOAA number: 12321 M1.1 2015-04-23 10:07:00 should be 12322 ; changing now.
There are 26 DONKI mistakes so far.


Case 3: The flare peak time is wrong in the DONKI database.

In [13]:
# create an empty array to hold row numbers to drop at the end
event_list_drops = []

ar_noaa = hek_results[:]["ar_noaanum"]
fc_noaa = hek_results[:]["fl_goescls"]

for i in range(events_list.shape[0]):
    # check if a particular DONKI flare peak time is also in the NOAA database
    peak_time_donki = events_list.loc[i,'peakTime']
    if not peak_time_donki in peak_times_noaa:
        active_region_number_donki = int(
            events_list.loc[i,'activeRegionNum'])
        flare_class_donki = events_list.loc[i,'classType']
        flare_class_indices = [j for j, x in enumerate(
            fc_noaa) if x == flare_class_donki]
        active_region_indices = [j for j, x in enumerate(
            ar_noaa) if x == active_region_number_donki]
        common_indices = list(
            set(flare_class_indices).intersection(active_region_indices))
        if common_indices:
            print("Messed up time:", int(events_list.loc[i,'activeRegionNum']), events_list.loc[i,'classType'],
                    events_list.loc[i,'peakTime'], "should be", peak_times_noaa[common_indices[0]], "; changing now.")
            events_list.loc[i,'peakTime'] = peak_times_noaa[common_indices[0]]
            number_of_donki_mistakes += 1
        if not common_indices:
            print("DONKI flare peak time",
                    events_list.loc[i,'peakTime'], "has no match; dropping row.")
            event_list_drops.append(i)
            number_of_donki_mistakes += 1
    
# Drop the rows for which the NOAA active region number and flare class associated with
# the messed-up flare peak time in the DONKI database has no match in the GOES flare database
events_list = events_list.drop(event_list_drops)
events_list = events_list.reset_index(drop=True)

# Create a list of corrected flare peak times
peak_times_donki = [events_list.loc[i,'peakTime']
                    for i in range(events_list.shape[0])]

print('There are', number_of_donki_mistakes, 'DONKI mistakes altogether.')

Messed up time: 11429 X1.1 2012-03-05 04:05:00 should be 2012-03-05 04:09:00.000 ; changing now.
DONKI flare peak time 2012-03-10 17:27:00 has no match; dropping row.
Messed up time: 11745 M5.0 2013-05-22 13:38:00 should be 2013-05-22 13:32:00.000 ; changing now.
Messed up time: 0 M1.1 2014-01-27 02:10:00 should be 2011-03-10 22:41:00.000 ; changing now.
DONKI flare peak time 2014-02-09 16:14:00 has no match; dropping row.
DONKI flare peak time 2014-05-06 22:09:00 has no match; dropping row.
Messed up time: 12127 M1.5 2014-08-01 18:12:00 should be 2014-08-01 18:13:00.000 ; changing now.
Messed up time: 12146 M2.0 2014-08-25 15:10:00 should be 2014-08-25 15:11:00.000 ; changing now.
DONKI flare peak time 2014-09-03 13:53:00 has no match; dropping row.
DONKI flare peak time 2014-09-09 00:28:00 has no match; dropping row.
Messed up time: 12172 M2.3 2014-09-23 23:15:00 should be 2014-09-23 23:16:00.000 ; changing now.
Messed up time: 12242 X1.8 2014-12-20 00:24:00 should be 2014-12-20 00:2

Now the "positive" labels have been collected and cleaned up. Next we need to get the feature data collected and cleaned.

In [14]:
answer = pd.read_csv(
    'http://jsoc.stanford.edu/doc/data/hmi/harpnum_to_noaa/all_harps_with_noaa_ars.txt', sep=' ')

In [15]:
timedelayvariable = 24

In [16]:
t_rec = [(events_list.loc[i,'peakTime'] - timedelta(hours=timedelayvariable)
            ).strftime('%Y.%m.%d_%H:%M_TAI') for i in range(events_list.shape[0])]

In [17]:
def get_the_jsoc_data(event_start,event_count, t_rec):
    """
    Parameters
    ----------
    event_start: Starting index
    event_count: number of events 
                 int

    t_rec:       list of times, one associated with each event in event_count
                 list of strings in JSOC format ('%Y.%m.%d_%H:%M_TAI')

    """

    catalog_data = []
    classification = []

    for i in range(event_count):

        print("=====", i, "=====")
        # next match NOAA_ARS to HARPNUM
        idx = answer[answer['NOAA_ARS'].str.contains(
            str(int(listofactiveregions[event_start+i])))]

        # if there's no HARPNUM, quit
        if (idx.empty == True):
            print('skip: there are no matching HARPNUMs for',
                  str(int(listofactiveregions[event_start+i])),flush=True)
            continue

        # construct jsoc_info queries and query jsoc database; we are querying for 25 keywords
        url = "http://jsoc.stanford.edu/cgi-bin/ajax/jsoc_info?ds=hmi.sharp_720s["+str(
            idx.HARPNUM.values[0])+"]["+t_rec[i]+"][? (CODEVER7 !~ '1.1 ') and (abs(OBS_VR)< 3500) and (QUALITY<65536) ?]&op=rs_list&key=USFLUX,MEANGBT,MEANJZH,MEANPOT,SHRGT45,TOTUSJH,MEANGBH,MEANALP,MEANGAM,MEANGBZ,MEANJZD,TOTUSJZ,SAVNCPP,TOTPOT,MEANSHR,AREA_ACR,R_VALUE,ABSNJZH"
        for attempt in range(5):
            try:
                response = requests.get(url,timeout=5)
                # ok = True
                break
            except requests.exceptions.Timeout:
                print('The sharp_720s request #{} timed out'.format(attempt))

        # if there's no response at this time, quit
        if response.status_code != 200:
            print('skip: cannot successfully get an http response',flush=True)
            continue

        # read the JSON output
        data = response.json()

        # if there are no data at this time, quit
        if data['count'] == 0:
            print('skip: there are no data for HARPNUM',
                  idx.HARPNUM.values[0], 'at time', t_rec[i],flush=True)
            continue

        # check to see if the active region is too close to the limb
        # we can compute the latitude of an active region in stonyhurst coordinates as follows:
        # latitude_stonyhurst = CRVAL1 - CRLN_OBS
        # for this we have to query the CEA series (but above we queried the other series as the CEA series does not have CODEVER5 in it)

        url = "http://jsoc.stanford.edu/cgi-bin/ajax/jsoc_info?ds=hmi.sharp_cea_720s["+str(
            idx.HARPNUM.values[0])+"]["+t_rec[i]+"][? (abs(OBS_VR)< 3500) and (QUALITY<65536) ?]&op=rs_list&key=CRVAL1,CRLN_OBS"
        #response = requests.get(url,timeout=240)
        # ok = False
        for attempt in range(5):
            try:
                response = requests.get(url,timeout=5)
                # ok = True
                break
            except requests.exceptions.Timeout:
                print('The sharp_cea_720s request #{} timed out'.format(attempt))
        
        # if there's no response at this time, quit
        if response.status_code != 200:
        # if not ok:
            print('skip: failed to find CEA JSOC data for HARPNUM',
                  idx.HARPNUM.values[0], 'at time', t_rec[i],flush=True)
            continue

        # read the JSON output
        latitude_information = response.json()

        # if there are no data at this time, quit
        if latitude_information['count'] == 0:
            print('skip: there are no data for HARPNUM',
                  idx.HARPNUM.values[0], 'at time', t_rec[i],flush=True)
            continue

        CRVAL1 = float(latitude_information['keywords'][0]['values'][0])
        CRLN_OBS = float(latitude_information['keywords'][1]['values'][0])
        if (np.absolute(CRVAL1 - CRLN_OBS) > 70.0):
            print('skip: latitude is out of range for HARPNUM',
                  idx.HARPNUM.values[0], 'at time', t_rec[i],flush=True)
            continue

        if ('MISSING' in str(data['keywords'])):
            print('skip: there are some missing keywords for HARPNUM',
                  idx.HARPNUM.values[0], 'at time', t_rec[i],flush=True)
            continue

        print('accept NOAA Active Region number', str(int(
            listofactiveregions[event_start+i])), 'and HARPNUM', idx.HARPNUM.values[0], 'at time', t_rec[i],flush=True)

        individual_flare_data = []
        for j in range(18):
            individual_flare_data.append(
                float(data['keywords'][j]['values'][0]))

        catalog_data.append(list(individual_flare_data))

        single_class_instance = [idx.HARPNUM.values[0], str(
            int(listofactiveregions[i])), listofgoesclasses[i], t_rec[i]]
        classification.append(single_class_instance)

    return catalog_data, classification

Phew! Data cleaning as per flares producing CMEs done! 
======================================================
Quite a process...
------------------

In [18]:
listofactiveregions = list(events_list['activeRegionNum'].values.flatten())
listofgoesclasses = list(events_list['classType'].values.flatten())

In [19]:
# Reading in JSOC data
positive_result = get_the_jsoc_data(0,events_list.shape[0], t_rec)

===== 0 =====
accept NOAA Active Region number 11158 and HARPNUM 377 at time 2011.02.14_01:56_TAI
===== 1 =====
skip: there are no data for HARPNUM 392 at time 2011.02.23_07:35_TAI
===== 2 =====
accept NOAA Active Region number 11166 and HARPNUM 401 at time 2011.03.06_14:30_TAI
===== 3 =====
accept NOAA Active Region number 11164 and HARPNUM 393 at time 2011.03.06_20:12_TAI
===== 4 =====
skip: there are no data for HARPNUM 415 at time 2011.03.07_03:58_TAI
===== 5 =====
accept NOAA Active Region number 11226 and HARPNUM 637 at time 2011.06.06_06:41_TAI
===== 6 =====
accept NOAA Active Region number 11261 and HARPNUM 750 at time 2011.08.02_13:48_TAI
===== 7 =====
accept NOAA Active Region number 11261 and HARPNUM 750 at time 2011.08.03_03:57_TAI
===== 8 =====
accept NOAA Active Region number 11283 and HARPNUM 833 at time 2011.09.06_22:38_TAI
===== 9 =====
skip: there are no data for HARPNUM 892 at time 2011.09.21_11:01_TAI
===== 10 =====
skip: there are no data for HARPNUM 892 at time 20

In [20]:
CME_data = positive_result[0]
positive_class = positive_result[1]
print("There are", len(CME_data), "CME events in the positive class.")

There are 51 CME events in the positive class.


In [21]:
# select peak times that belong to both classes
all_peak_times = np.array([(hek_results[i]['event_peaktime'])
                        for i in range(len(hek_results))])

negative_class_possibilities = []
counter_positive = 0
counter_negative = 0
for i in range(len(hek_results)):
    this_peak_time = all_peak_times[i]
    if (this_peak_time in peak_times_donki):
        counter_positive += 1
    else:
        counter_negative += 1
        this_instance = [hek_results[i]['ar_noaanum'],
                            hek_results[i]['fl_goescls'], hek_results[i]['event_peaktime']]
        negative_class_possibilities.append(this_instance)
print("There are", counter_positive, "events in the positive class.")
print("There are", counter_negative, "events in the negative class.")

There are 111 events in the positive class.
There are 476 events in the negative class.


In [22]:
t_recn = np.array([(negative_class_possibilities[i][2] - timedelta(hours=timedelayvariable)
                   ).strftime('%Y.%m.%d_%H:%M_TAI') for i in range(len(negative_class_possibilities))])

In [23]:
listofactiveregions = list(
    negative_class_possibilities[i][0] for i in range(counter_negative))
listofgoesclasses = list(
    negative_class_possibilities[i][1] for i in range(counter_negative))

In [24]:
negative_result = get_the_jsoc_data(0,counter_negative, t_recn)

===== 0 =====
accept NOAA Active Region number 11069 and HARPNUM 8 at time 2010.05.04_17:19_TAI
===== 1 =====
skip: there are no data for HARPNUM 54 at time 2010.06.11_00:57_TAI
===== 2 =====
accept NOAA Active Region number 11112 and HARPNUM 211 at time 2010.10.15_19:12_TAI
===== 3 =====
skip: latitude is out of range for HARPNUM 245 at time 2010.11.03_23:58_TAI
===== 4 =====
skip: latitude is out of range for HARPNUM 245 at time 2010.11.03_23:58_TAI
===== 5 =====
skip: latitude is out of range for HARPNUM 245 at time 2010.11.05_15:36_TAI
===== 6 =====
accept NOAA Active Region number 11149 and HARPNUM 345 at time 2011.01.27_01:03_TAI
===== 7 =====
accept NOAA Active Region number 11153 and HARPNUM 362 at time 2011.02.08_01:31_TAI
===== 8 =====
accept NOAA Active Region number 11158 and HARPNUM 377 at time 2011.02.12_17:38_TAI
===== 9 =====
accept NOAA Active Region number 11158 and HARPNUM 377 at time 2011.02.13_17:26_TAI
===== 10 =====
accept NOAA Active Region number 11161 and HARP

In [25]:
no_CME_data = negative_result[0]
negative_class = negative_result[1]
print("There are", len(no_CME_data), "no-CME events in the negative class.")

There are 309 no-CME events in the negative class.


Save the data for analysis
--------------------------

In [26]:
np.savetxt('/coursedata/A2/CMEs_24_BI.csv', CME_data, delimiter=',')
np.savetxt('/coursedata/A2/noCMEs_24_BI.csv', no_CME_data, delimiter=',')

OSError: [Errno 30] Read-only file system: '/coursedata/A2/CMEs_24_BI.csv'